Step 1. Import & merge data

In [1]:
import pandas as pd

# Load all CSVs
df1 = pd.read_csv("nba_savant 2015-2016 a-h.csv")
df2 = pd.read_csv("nba_savant 2015-2016 i-m.csv")
df3 = pd.read_csv("nba_savant 2015-2016 n-z.csv")

# Merge into one dataset
df = pd.concat([df1, df2, df3], ignore_index=True)

print("Shape:", df.shape)
df.head()


Shape: (145986, 22)


,name,team_name,game_date,season,espn_player_id,team_id,espn_game_id,period,minutes_remaining,seconds_remaining,...,shot_type,shot_distance,opponent,x,y,dribbles,touch_time,defender_name,defender_distance,shot_clock
0,Clint Capela,Houston Rockets,2016-01-24,2015,3102529.0,1610612745,400828552,4,1,26,...,2PT Field Goal,0,Dallas Mavericks,0,1,0,0.0,NaN,0.0,0.0
1,Clint Capela,Houston Rockets,2015-10-28,2015,3102529.0,1610612745,400827897,2,4,50,...,2PT Field Goal,3,Denver Nuggets,-25,31,0,0.0,"Mudiay, Emmanuel",2.7,12.9
2,Tim Hardaway Jr,Atlanta Hawks,2016-01-09,2015,2528210.0,1610612737,400828438,1,2,0,...,2PT Field Goal,0,Chicago Bulls,0,1,0,0.0,"Butler, Jimmy",4.9,20.4
3,Andre Drummond,Detroit Pistons,2015-10-28,2015,6585.0,1610612765,400827894,3,2,28,...,2PT Field Goal,3,Utah Jazz,18,26,0,0.0,"Favors, Derrick",3.3,0.0
4,Kenneth Faried,Denver Nuggets,2016-02-08,2015,6433.0,1610612743,400828665,2,4,1,...,2PT Field Goal,0,Brooklyn Nets,0,1,0,0.0,NaN,0.0,0.0


### Step 2. Target variable

In [2]:
y = df["shot_made_flag"]   # 1 = made, 0 = missed


### Step 3. Feature selection

Numeric features:
period, minutes_remaining, seconds_remaining, shot_distance, x, y, dribbles, touch_time, defender_distance, shot_clock

Categorical features (need encoding):
name, team_name, opponent, action_type, shot_type, defender_name

In [3]:
categorical_cols = ["name", "team_name", "opponent", 
                    "action_type", "shot_type", "defender_name"]

numeric_cols = ["period", "minutes_remaining", "seconds_remaining",
                "shot_distance", "x", "y", "dribbles", "touch_time",
                "defender_distance", "shot_clock"]


### Step 4. Encode categorical variables

In [4]:
df["player_name"] = df["name"]   # save original name column
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

features = numeric_cols + [col for col in df.columns if any(c in col for c in categorical_cols)]
X = df[features]


### Step 5. Train/test split

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


### Step 6. Train baseline models

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

models = {
    # "Logistic": LogisticRegression(max_iter=10000),
    # "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "GradientBoost": GradientBoostingClassifier()
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    print(f"{name}: acc={accuracy_score(y_test, y_pred):.3f}, auc={roc_auc_score(y_test, y_proba):.3f}")


GradientBoost: acc=0.659, auc=0.696


### Step 7. Expected points per shot

In [11]:
# Choose best model (e.g., GradientBoost)
best_model = models["GradientBoost"]

# Add predicted make probability
df["make_prob"] = best_model.predict_proba(X)[:,1]

# Shot points (2 or 3)
df["shot_points"] = df["shot_type_3PT Field Goal"]*3 + (1 - df["shot_type_3PT Field Goal"])*2

# Expected value
df["expected_points"] = df["make_prob"] * df["shot_points"]

# Actual points
df["actual_points"] = df["shot_made_flag"] * df["shot_points"]

df.head()


,game_date,season,espn_player_id,team_id,espn_game_id,period,minutes_remaining,seconds_remaining,shot_made_flag,shot_distance,...,"defender_name_Young, James","defender_name_Young, Joe","defender_name_Young, Nick","defender_name_Young, Thaddeus","defender_name_Zeller, Cody","defender_name_Zeller, Tyler",make_prob,shot_points,expected_points,actual_points
0,2016-01-24,2015,3102529.0,1610612745,400828552,4,1,26,1,0,...,False,False,False,False,False,False,0.844928,2,1.689856,2
1,2015-10-28,2015,3102529.0,1610612745,400827897,2,4,50,1,3,...,False,False,False,False,False,False,0.535350,2,1.070700,2
2,2016-01-09,2015,2528210.0,1610612737,400828438,1,2,0,1,0,...,False,False,False,False,False,False,0.900547,2,1.801094,2
3,2015-10-28,2015,6585.0,1610612765,400827894,3,2,28,1,3,...,False,False,False,False,False,False,0.540836,2,1.081673,2
4,2016-02-08,2015,6433.0,1610612743,400828665,2,4,1,1,0,...,False,False,False,False,False,False,0.843001,2,1.686002,2


### Step 8. Player-level “luck”

In [12]:
# later you can group by "player_name"
player_summary = df.groupby("player_name").agg(
    exp_points=("expected_points", "sum"),
    actual_points=("actual_points", "sum"),
    shots=("shot_made_flag", "count")
)

player_summary["luck"] = player_summary["actual_points"] - player_summary["exp_points"]
player_summary.sort_values("luck", ascending=False).head(10)


KeyError: 'name'